In [ ]:
# Histogram of track lifetimes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys 
import os
import seaborn as sns
import matplotlib.ticker as ticker
import zarr 
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

from intensity_time_plots import filter_track_ids_by_length_ranges, random_track_ids
from intensity_time_plots import intensity_time_plot, createBufferForLifetimeCohort, intensity_time_plot_onetrack
from intensity_time_plots import createBufferForLifetimeCohort_normalized, cumulative_plots, cumulative_plots_ax


%load_ext autoreload
%autoreload 2

In [ ]:
base_dir = r'Z:\Abhi\LLSM_Analysis'
input_file_directory = 'controlOS_analysis/'

# # If a folder named 'datasets' doesn't exist in base_dir + input_file_directory, it will be created
# if not os.path.exists(os.path.join(base_dir, input_file_directory, 'datasets')):
#     os.makedirs(os.path.join(base_dir, input_file_directory, 'datasets'))

zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_file_directory)

input_directory_all_tracks_full = os.path.join(base_dir, input_file_directory) + 'datasets/track_df_cleaned_final_full.pkl'


input_directory_tracks = 'datasets'
input_directory_filtered = 'filtered_tracks_final.pkl'
input_directory_full_filtered_tracks = (os.path.join(base_dir, input_file_directory, 
                                                    input_directory_tracks, input_directory_filtered))

In [ ]:
track_df = pd.read_pickle(input_directory_all_tracks_full)
filtered_tracks = pd.read_pickle(input_directory_full_filtered_tracks)
z2 = zarr.open(zarr_full_path, mode='r')

In [ ]:
# Number of frames in the dataset, important for plotting montages later on
max_frames = z2.shape[0]
print(f'Max frames in the dataset: {max_frames}')

In [ ]:
channels_to_plot = [True, True]
triple_positive = filtered_tracks[(filtered_tracks['channel1_positive'] == channels_to_plot[0]) & (filtered_tracks['channel2_positive'] == channels_to_plot[1])]
dynamin_negative_arpc3_negative = filtered_tracks[(filtered_tracks['channel1_positive'] == False) & 
                                                  (filtered_tracks['channel2_positive'] == False)]
low_ARPC3 = triple_positive[(triple_positive['C1_adjusted_voxel_sum_peak'] > 2500) & 
                           (triple_positive['C1_adjusted_voxel_sum_peak'] < 4000)]
high_ARPC3 = triple_positive[(triple_positive['C1_adjusted_voxel_sum_peak'] > 4000)]
# low_ARPC3['track_id']
# high_ARPC3['track_id']
#Conver dynamin positive ARPC3 negative tracks to a list
dn_an = dynamin_negative_arpc3_negative['track_id'].tolist()
# dn_an[333]

In [ ]:
high_ARPC3[high_ARPC3['membrane_region'] == 'Basal']['track_id']

In [ ]:
ID = 36541
# triple_positive[triple_positive['track_id'] == ID]['C1_adjusted_voxel_sum_peak']

In [ ]:
# track_df[track_df['track_id'] == ID][['frame', 'c2_voxel_sum_adjusted', 'mu_x', 'mu_y', 'mu_z']]

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# import numpy as np
# from matplotlib.colors import LinearSegmentedColormap

# def plot_track_3d(df, track_id):
#     """
#     Plot the 3D position of a specific track over time.
    
#     Parameters:
#     df (pandas.DataFrame): DataFrame containing track data with multiple time points
#     track_id (int): The track_id to plot
#     """
#     # Filter data for the specific track
#     track_data = df[df['track_id'] == track_id].copy()
    
#     if track_data.empty:
#         print(f"No data found for track_id {track_id}")
#         return
    
#     # Get the position data - assuming these are arrays stored in a single row
#     x = track_data['mu_x'].values[0]
#     y = track_data['mu_y'].values[0]
#     z = track_data['mu_z'].values[0]
    
#     # Convert to numpy arrays if they aren't already
#     if not isinstance(x, np.ndarray):
#         x = np.array(x)
#     if not isinstance(y, np.ndarray):
#         y = np.array(y)
#     if not isinstance(z, np.ndarray):
#         z = np.array(z)
    
#     # Create time points
#     time_points = np.arange(len(x))
    
#     # Create the 3D plot
#     fig = plt.figure(figsize=(15, 12))
#     ax = fig.add_subplot(111, projection='3d')
    
#     # Plot the trajectory with color gradient representing time
#     scatter = ax.scatter(x, y, z, c=time_points, cmap='viridis', s=50, alpha=0.6)
    
#     # Plot the line connecting points
#     ax.plot(x, y, z, 'b-', alpha=0.3)
    
#     # Mark start and end points
#     ax.scatter(x[0], y[0], z[0], color='green', s=100, marker='o', label='Start')
#     ax.scatter(x[-1], y[-1], z[-1], color='red', s=100, marker='s', label='End')
    
#     # Calculate axis ranges for equal scaling
#     x_range = [x.min(), x.max()]
#     y_range = [y.min(), y.max()]
#     z_range = [z.min(), z.max()]
    
#     # Find the maximum range
#     max_range = max(x_range[1] - x_range[0], 
#                     y_range[1] - y_range[0], 
#                     z_range[1] - z_range[0])
    
#     # Calculate centers
#     x_center = (x_range[1] + x_range[0]) / 2
#     y_center = (y_range[1] + y_range[0]) / 2
#     z_center = (z_range[1] + z_range[0]) / 2
    
#     # Set equal aspect ratio
#     ax.set_xlim([x_center - max_range/2, x_center + max_range/2])
#     ax.set_ylim([y_center - max_range/2, y_center + max_range/2])
#     ax.set_zlim([z_center - max_range/2, z_center + max_range/2])
    
#     # # Add colorbar for time
#     # cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
#     # cbar.set_label('Time Step', rotation=270, labelpad=15, fontweight='bold')

#     # Set labels and title with bold formatting
#     ax.set_xlabel('X Position (voxels)', fontweight='bold', labelpad=10)
#     ax.set_ylabel('Y Position (voxels)', fontweight='bold', labelpad=10)
#     ax.set_zlabel('Z Position (voxels)', fontweight='bold', labelpad=10)
#     # ax.set_title(f'3D Track Position for Track ID: {track_id}', fontweight='bold')

#     # Position legend at upper center (similar to 2D plots)
#     ax.legend(loc='upper left')

#     # Set tick parameters for all axes (X, Y, Z)
#     # X-axis tick parameters
#     ax.tick_params(axis='x', reset=True, direction='out', 
#                 color='black', length=4, labelsize=12)
#     ax.tick_params(axis='x', which='both', top=False)

#     # Y-axis tick parameters
#     ax.tick_params(axis='y', reset=True, direction='out', 
#                 color='black', length=4, labelsize=12)
#     ax.tick_params(axis='y', which='both', right=False)
#     # for label in ax.get_yticklabels():
#     #     label.set_rotation(45)

#     # Z-axis tick parameters
#     ax.tick_params(axis='z', reset=True, direction='out', 
#                 color='black', length=4, labelsize=12)
#     # ax.tick_params(axis='z', which='both', top=False)
#     # for label in ax.get_xticklabels():
#     #     label.set_rotation(45)
    
#     # Set the viewing angle
#     ax.view_init(elev=20, azim=45)
    
#     plt.tight_layout()
#     plt.show()

In [ ]:
# plot_track_3d(filtered_tracks, ID)  # Replace with the desired track_id

In [ ]:
def plot_track_3d_enhanced(df, track_id, save_plot=False):
    """
    Plot the 3D position of a specific track over time with enhanced aesthetics.
   
    Parameters:
    df (pandas.DataFrame): DataFrame containing track data with multiple time points
    track_id (int): The track_id to plot
    """
    # Filter data for the specific track
    track_data = df[df['track_id'] == track_id].copy()
   
    if track_data.empty:
        print(f"No data found for track_id {track_id}")
        return
   
    # Get the position data
    x = track_data['mu_x'].values[0]
    y = track_data['mu_y'].values[0]
    z = track_data['mu_z'].values[0]
   
    # Convert to numpy arrays if they aren't already
    if not isinstance(x, np.ndarray):
        x = np.array(x)
    if not isinstance(y, np.ndarray):
        y = np.array(y)
    if not isinstance(z, np.ndarray):
        z = np.array(z)
   
    # Create time points normalized to [0, 1]
    time_points = np.linspace(0, 1, len(x))
   
    # Set style and create figure with better aesthetics
    plt.style.use('seaborn-v0_8-whitegrid')  # Modern clean style
    fig = plt.figure(figsize=(10, 8.75), facecolor='white')
    ax = fig.add_subplot(111, projection='3d', facecolor='white')
    
    # Custom colormap - modern gradient from cool to warm
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    n_bins = 256
    custom_cmap = LinearSegmentedColormap.from_list('trajectory', colors, N=n_bins)
    
    # Plot trajectory points with size variation (larger = more recent)
    sizes = np.linspace(30, 80, len(x))  # Gradually increasing size
    scatter = ax.scatter(x, y, z, c=time_points, cmap=custom_cmap, 
                        s=sizes, alpha=0.8, edgecolors='white', linewidths=0.5)
   
    # Plot connecting line with gradient effect
    for i in range(len(x)-1):
        ax.plot([x[i], x[i+1]], [y[i], y[i+1]], [z[i], z[i+1]], 
                color=custom_cmap(time_points[i]), alpha=0.4, linewidth=2)
   
    # Enhanced start and end markers
    ax.scatter(x[0], y[0], z[0], color='#00C851', s=150, marker='o', 
              label='Start', edgecolors='white', linewidths=2, alpha=0.9)
    ax.scatter(x[-1], y[-1], z[-1], color='#FF4444', s=150, marker='s', 
              label='End', edgecolors='white', linewidths=2, alpha=0.9)
   
    # Calculate axis ranges for equal scaling
    x_range = [x.min(), x.max()]
    y_range = [y.min(), y.max()]
    z_range = [z.min(), z.max()]
   
    # Add padding to ranges
    padding = 0.1
    x_pad = (x_range[1] - x_range[0]) * padding
    y_pad = (y_range[1] - y_range[0]) * padding
    z_pad = (z_range[1] - z_range[0]) * padding
    
    x_range = [x_range[0] - x_pad, x_range[1] + x_pad]
    y_range = [y_range[0] - y_pad, y_range[1] + y_pad]
    z_range = [z_range[0] - z_pad, z_range[1] + z_pad]
   
    # Find the maximum range
    max_range = max(x_range[1] - x_range[0],
                    y_range[1] - y_range[0],
                    z_range[1] - z_range[0])
   
    # Calculate centers
    x_center = (x_range[1] + x_range[0]) / 2
    y_center = (y_range[1] + y_range[0]) / 2
    z_center = (z_range[1] + z_range[0]) / 2
   
    # Set equal aspect ratio
    ax.set_xlim([x_center - max_range/2, x_center + max_range/2])
    ax.set_ylim([y_center - max_range/2, y_center + max_range/2])
    ax.set_zlim([z_center - max_range/2, z_center + max_range/2])
   
    # Enhanced colorbar
    cbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.8, aspect=20)
    cbar.set_label('Trajectory Progress', rotation=270, labelpad=20, 
                   fontsize=14, fontweight='bold')
    cbar.ax.tick_params(labelsize=10)
    
    # Modern styling for axes
    ax.set_xlabel('X Position (voxels)', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_ylabel('Y Position (voxels)', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_zlabel('Z Position (voxels)', fontsize=14, fontweight='bold', labelpad=10)
    
    # # Enhanced title
    # ax.set_title(f'3D Trajectory Visualization\nTrack ID: {track_id}', 
    #             fontsize=16, fontweight='bold', pad=20)
    
    # Improved legend with better styling
    legend = ax.legend(loc='upper left', frameon=True, fancybox=False, 
                      shadow=False, fontsize=12, bbox_to_anchor=(0, 1))
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_alpha(0.9)
    for text in legend.get_texts():
        text.set_fontweight('bold')
    
    # Enhanced grid and background
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    
    # Make pane edges more subtle
    ax.xaxis.pane.set_edgecolor('gray')
    ax.yaxis.pane.set_edgecolor('gray')
    ax.zaxis.pane.set_edgecolor('gray')
    ax.xaxis.pane.set_alpha(0.1)
    ax.yaxis.pane.set_alpha(0.1)
    ax.zaxis.pane.set_alpha(0.1)
    ax.invert_xaxis()  # Invert x-axis for better depth perception
    ax.invert_yaxis()  # Invert y-axis for better depth perception
    
    # Enhanced tick parameters
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_tick_params(labelsize=11, colors='#333333')
        
    # Optimal viewing angle for better depth perception
    ax.view_init(elev=25, azim=135)
   
    plt.tight_layout()
    if save_plot:
        plt.savefig(f'track_{track_id}_3d.png', dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
plot_track_3d_enhanced(filtered_tracks, ID, save_plot = False)  # Replace with the desired track_id

In [ ]:
value_to_plot = 'voxel_sum_adjusted'
channel3_name = 'AP2'
channel2_name = 'Dynamin2'
channel1_name = 'ArpC3'

# set 'normalized = False' to plot raw intensities

intensity_time_plot_onetrack(dataframe = track_df, tracks_to_plot = ID, 
intensity_to_plot = [f'c3_{value_to_plot}',f'c2_{value_to_plot}', f'c1_{value_to_plot}'], track_id_col_name = 'track_id', 
frame_col_name = 'frame', channels_to_plot = 3, legend_values = [channel3_name, channel2_name, channel1_name],
line_colors = ['magenta', 'lime', 'blue'], graph_title = f'{value_to_plot}', normalized=False, save_plot = False)

In [ ]:
crop_size_z = 4
crop_size_xy = 3  # Half-width of the square patch

time = list(track_df[track_df['track_id'] == ID]['frame'])
print(f'Time points for track {ID}: {time}')
min_frame = min(time)
max_frame = max(time)

# Get coordinates per frame for this track
x_coord = filtered_tracks[filtered_tracks['track_id'] == ID]['mu_x'].values[0]
y_coord = filtered_tracks[filtered_tracks['track_id'] == ID]['mu_y'].values[0]
z_coord = filtered_tracks[filtered_tracks['track_id'] == ID]['mu_z'].values[0]

# Convert to integer coordinate tuples
coordinates_real = list(zip(x_coord, y_coord, z_coord))
coordinates_real = [tuple(int(val) for val in tup) for tup in coordinates_real]


# Prepare stacks for each channel
stack_arpc3, stack_dnm, stack_ap2, stack_merge = [], [], [], []

for t, (x, y, z) in zip(time, coordinates_real):
    # Extract patches (z, y, x) for all three channels, which includes a background strip
    arpc3_patch = z2[t, 0, (z-crop_size_z):(z+crop_size_z+1), (y-crop_size_xy):(y+crop_size_xy+1), (x-crop_size_xy):(x+crop_size_xy+1)]
    dnm_patch   = z2[t, 1, (z-crop_size_z):(z+crop_size_z+1), (y-crop_size_xy):(y+crop_size_xy+1), (x-crop_size_xy):(x+crop_size_xy+1)]
    ap2_patch   = z2[t, 2, (z-crop_size_z):(z+crop_size_z+1), (y-crop_size_xy):(y+crop_size_xy+1), (x-crop_size_xy):(x+crop_size_xy+1)]

    # Max intensity projection
    arpc3_proj = np.max(arpc3_patch, axis=0)
    dnm_proj   = np.max(dnm_patch, axis=0)
    ap2_proj   = np.max(ap2_patch, axis=0)

    # Normalize for display
    arpc3_norm = arpc3_proj / np.max(arpc3_proj) if np.max(arpc3_proj) > 0 else arpc3_proj
    dnm_norm   = dnm_proj   / np.max(dnm_proj)   if np.max(dnm_proj)   > 0 else dnm_proj
    ap2_norm   = ap2_proj   / np.max(ap2_proj)   if np.max(ap2_proj)   > 0 else ap2_proj

    # Add to individual grayscale channel strips
    stack_arpc3.append(arpc3_norm)
    stack_dnm.append(dnm_norm)
    stack_ap2.append(ap2_norm)

    # Merge: RFP (red), GFP (green), HALO (blue)
    rgb = np.stack([
        ap2_norm,    # Red: AP2-RFP
        dnm_norm,    # Green: DNM2-GFP
        arpc3_norm   # Blue: ARPC3-HALO
    ], axis=-1)
    stack_merge.append(rgb)

# Concatenate each channel across time
ap2_strip    = np.hstack(stack_ap2)
dnm_strip    = np.hstack(stack_dnm)
arpc3_strip  = np.hstack(stack_arpc3)
merge_strip  = np.hstack(stack_merge)

# Plot the 4 rows
fig, ax = plt.subplots(4, 1, figsize=(len(stack_ap2), 4), dpi=600)

ax[0].imshow(ap2_strip, cmap='gray')
ax[0].set_title('AP2', fontsize=12, fontweight='bold')

ax[1].imshow(dnm_strip, cmap='gray')
ax[1].set_title('Dynamin2', fontsize=12, fontweight='bold')

ax[2].imshow(arpc3_strip, cmap='gray')
ax[2].set_title('ArpC3', fontsize=12, fontweight='bold')

ax[3].imshow(merge_strip)
ax[3].set_title('Merge', fontsize=12, fontweight='bold')


# ax[0].imshow(rfp_strip, cmap='Reds')        # or 'magenta' 
# ax[0].set_title('AP2', color = 'red', fontsize = 12, fontweight='bold')
# ax[1].imshow(dnm_strip, cmap='Greens')       # or custom cyan
# ax[1].set_title('Dynamin2', color = 'green', fontsize = 12, fontweight='bold')
# ax[2].imshow(arpc3_strip, cmap='Blues')    # or custom yellow
# ax[2].set_title('ArpC3', color = 'blue', fontsize = 12, fontweight='bold')
# ax[3].imshow(merge_strip)  # Your RGB merge
# ax[3].set_title('Merge', color = 'black', fontsize = 12, fontweight='bold')

for a in ax:
    a.axis('off')

plt.tight_layout()

save_plot = False  # Set this to True if you want to save the plot
if save_plot:
    filename = f'kymograph_track_{ID}.png'
    plt.savefig(filename, dpi=600, bbox_inches='tight', facecolor='white')

plt.show()

In [ ]:
# Montages with a few (3) frames before and after the track's time points

crop_size_z = 4
crop_size_xy = 3  # Half-width of the square patch

time = list(track_df[track_df['track_id'] == ID]['frame'])
# Print max frames in the dataset
print(f'Max frames in the dataset: {max_frames}')
print(f'Time points for track {ID}: {time}')
# Extend time range by 3 frames before and after the track's time points
# This ensures we have a buffer around the track for visualization
extended_time = list(range(max(0, min(time) - 3), min((max_frames-1), max(time) + 3) + 1)) if time else []
# Drop values in extended_time that are between min(time) and max(time), but not in time. This takes care 
# of missing frames in the track.
extended_time = [frame for frame in extended_time if frame < min(time) or frame > max(time) or frame in time]
print(f'Extended time points for track {ID}: {extended_time}')

# Get coordinates per frame for this track
x_coord = filtered_tracks[filtered_tracks['track_id'] == ID]['mu_x'].values[0]
y_coord = filtered_tracks[filtered_tracks['track_id'] == ID]['mu_y'].values[0]
z_coord = filtered_tracks[filtered_tracks['track_id'] == ID]['mu_z'].values[0]

# Extend coordinates to match new time range
extended_x = []
extended_y = []
extended_z = []

min_frame = min(time)
max_frame = max(time)
for frame in extended_time:
    if frame < min_frame:
        # Use first coordinate for prior frames
        extended_x.append(x_coord.iloc[0])
        extended_y.append(y_coord.iloc[0])
        extended_z.append(z_coord.iloc[0])
    elif frame > max_frame:
        # Use last coordinate for future frames
        extended_x.append(x_coord.iloc[-1])
        extended_y.append(y_coord.iloc[-1])
        extended_z.append(z_coord.iloc[-1])
    else:
        # Use actual coordinate for existing frames, skip if not in time
        frame_idx = time.index(frame) # if frame in time else None
        # if frame_idx is None: #skip if frame is not in time
        #     continue
        extended_x.append(x_coord.iloc[frame_idx])
        extended_y.append(y_coord.iloc[frame_idx])
        extended_z.append(z_coord.iloc[frame_idx])

# Convert to integer coordinate tuples
coordinates = list(zip(x_coord, y_coord, z_coord))
print(f'Coordinates for track {ID}: {coordinates}')
coordinates_extended = list(zip(extended_x, extended_y, extended_z))
print(f'Extended coordinates for track {ID}: {coordinates_extended}')
coordinates_extended = [tuple(int(val) for val in tup) for tup in coordinates_extended]


# Prepare stacks for each channel
stack_ap2, stack_dnm, stack_arpc3, stack_merge = [], [], [], []

for t, (x, y, z) in zip(extended_time, coordinates_extended):
    print(f'Processing frame {t} with coordinates (x={x}, y={y}, z={z})')
    # Extract patches (z, y, x) for all three channels
    arpc3_patch = z2[t, 0, (z-crop_size_z):(z+crop_size_z+1), (y-crop_size_xy):(y+crop_size_xy+1), (x-crop_size_xy):(x+crop_size_xy+1)]
    dnm_patch   = z2[t, 1, (z-crop_size_z):(z+crop_size_z+1), (y-crop_size_xy):(y+crop_size_xy+1), (x-crop_size_xy):(x+crop_size_xy+1)]
    ap2_patch   = z2[t, 2, (z-crop_size_z):(z+crop_size_z+1), (y-crop_size_xy):(y+crop_size_xy+1), (x-crop_size_xy):(x+crop_size_xy+1)]

    # Max intensity projection
    arpc3_proj = np.max(arpc3_patch, axis=0)
    dnm_proj   = np.max(dnm_patch, axis=0)
    ap2_proj   = np.max(ap2_patch, axis=0)

    # Normalize for display
    arpc3_norm = arpc3_proj / np.max(arpc3_proj) if np.max(arpc3_proj) > 0 else arpc3_proj
    dnm_norm   = dnm_proj   / np.max(dnm_proj)   if np.max(dnm_proj)   > 0 else dnm_proj
    ap2_norm   = ap2_proj   / np.max(ap2_proj)   if np.max(ap2_proj)   > 0 else ap2_proj

    # Add to individual grayscale channel strips
    stack_arpc3.append(arpc3_norm)
    stack_dnm.append(dnm_norm)
    stack_ap2.append(ap2_norm)

    # # Add to individual grayscale channel strips
    # stack_arpc3.append(arpc3_proj)
    # stack_dnm.append(dnm_proj)
    # stack_rfp.append(rfp_proj)

    # Merge: RFP (red), GFP (green), HALO (blue)
    rgb = np.stack([
        ap2_norm,    # Red: AP2-RFP
        dnm_norm,    # Green: DNM2-GFP
        arpc3_norm   # Blue: ARPC3-HALO
    ], axis=-1)
    stack_merge.append(rgb)

    # # Merge: RFP (red), GFP (green), HALO (blue)
    # rgb = np.stack([
    #     rfp_proj,    # Red: AP2-RFP
    #     dnm_proj,    # Green: DNM2-GFP
    #     arpc3_proj   # Blue: ARPC3-HALO
    # ], axis=-1)
    # stack_merge.append(rgb)

# Concatenate each channel across time
ap2_strip    = np.hstack(stack_ap2)
dnm_strip    = np.hstack(stack_dnm)
arpc3_strip  = np.hstack(stack_arpc3)
merge_strip  = np.hstack(stack_merge)

# Adding arrow positions for the extended time range
arrow_positions = []
for i, frame in enumerate(extended_time):
    if frame not in time:  # Frame is extended, not original
        arrow_positions.append(i * (2*crop_size_xy + 1) + crop_size_xy)  # Center of each patch

# Plot the 4 rows
fig, ax = plt.subplots(4, 1, figsize=(len(stack_ap2), 4), dpi=600)

strips = [ap2_strip, dnm_strip, arpc3_strip, merge_strip]
titles = ['AP2', 'Dynamin2', 'ArpC3', 'Merge']
cmaps = ['gray', 'gray', 'gray', None]

for i, (strip, title, cmap) in enumerate(zip(strips, titles, cmaps)):
   ax[i].imshow(strip, cmap=cmap)
   for pos in arrow_positions:
       ax[i].annotate('', xy=(pos, -0.5), xytext=(pos, 0), 
                      arrowprops=dict(arrowstyle='-|>', color='red', lw=2))
   ax[i].set_title(title, fontsize=12, fontweight='bold')

for a in ax:
    a.axis('off')

plt.tight_layout()

save_plot = False  # Set this to True if you want to save the plot
if save_plot:
    filename = f'kymograph_track_{ID}.png'
    plt.savefig(filename, dpi=600, bbox_inches='tight', facecolor='white')

plt.show()
